In [1]:
from microscope_calibration.common.model import Parameters4DSTEM, DescanError, trace, PixelYX, Model4DSTEM, Ray, symbol_maker, sympy_equals, scale_rotate_flip

from temgym_core.run import run_iter

import jax
import jax.numpy as jnp
import numpy as np
import sympy as sym
from sympy import S
from types import ModuleType

In [2]:
from sympy.abc import a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, s, t, u, v, w, x
parms= Parameters4DSTEM(
    overfocus=0.01*a,
    scan_pixel_pitch=1e-6*b,
    scan_center=PixelYX(0.1*c, 0.2*d),
    scan_rotation=0.01*e,
    camera_length=1.0*f,
    detector_pixel_pitch=50e-6*g,
    detector_center=PixelYX(0.1*h, 0.1*i),
    semiconv=1e-3*j,  # radian
    flip_factor=1.0*k,
    descan_error=DescanError(pxo_pxi=l, pxo_pyi=m, pyo_pxi=n, pyo_pyi=o, sxo_pxi=p, sxo_pyi=q, syo_pxi=r, syo_pyi=s, offpxi=t, offpyi=u, offsxi=v, offsyi=w),
)

In [3]:
def make_a_function(a: str):
    def f():
        return f'hello {a}'
    return f

In [4]:
f = make_a_function('Margarita')

In [5]:
f()

'hello Margarita'

In [6]:
def scale(factor):
    return sym.eye(2) * factor


def rotate(radians):
    return sym.rot_givens(0, 1, radians, dim=2)


# The flip_factor is introduced to make it differentiable
def flip_y(flip_factor: sym.Float = sym.S(-1.0)):
    return sym.Matrix([[flip_factor, 0], [0, 1]])


def identity():
    return sym.eye(2)

In [7]:
def scale_rotate_flip(mat: sym.Matrix):
    """
    Deconstruct a matrix generated with scale() @ rotate() @ flip_y()
    into the individual parameters
    """
    scale_y = mat[:, 0].norm()
    scale_x = mat[:, 1].norm()
    
    if not scale_x.equals(scale_y):
        raise ValueError(f"y scale {scale_y} and x scale {scale_x} are different.")

    scan_rot_flip = mat / scale_y

    # 2D cross product
    flip_factor = (
        scan_rot_flip[0, 0] * scan_rot_flip[1, 1]
        - scan_rot_flip[0, 1] * scan_rot_flip[1, 0]
    )
    # undo flip_y
    rot = scan_rot_flip.copy()
    rot[:, 0] = rot[:, 0] * flip_factor
    rot = sym.simplify(rot)

    angle1 = sym.atan2(-rot[1, 0], rot[0, 0])
    angle2 = sym.atan2(rot[0, 1], rot[1, 1])

    # So far not reached in tests since inconsistencies are caught as shear before
    if not sym.Matrix([sym.sin(angle1), sym.cos(angle1)]).equals(sym.Matrix([sym.sin(angle2), sym.cos(angle2)])):
        raise ValueError(
            f"Rotation angle 1 {angle1} and rotation angle 2 {angle2} are inconsistent."
        )

    if sym.Or(sym.And(mat[0,0].equals(mat[1,1]), mat[0,1].equals(-mat[1,0])), sym.And(mat[0,0].equals(-mat[1,1]), mat[0,1].equals(mat[1,0]))) is not sym.S.true:
        raise ValueError(f"y scale {scale_y} and x scale {scale_x} are different or rotation angle 1 {angle1} and rotation angle 2 {angle2} are inconsistent.")
        
    return (scale_y, angle1, flip_factor)

In [8]:
a, b, c, d = sym.symbols('a b c d', real=True)
expr = sym.sqrt(a**2+c**2)-sym.sqrt(b**2+d**2)
sol_c = sym.solve(expr, c)[0]
sol_d = sym.solve(expr, d)[0]
expr_subs = expr.subs(c, sol_c).subs(d, sol_d)
expr_subs
#scale_rotate_flip(sym.Matrix([[a,b],[b,a]]))

0

In [9]:
scale_rotate_flip(sym.Matrix([[0,0],[0,0]]))
sym.And(a.equals(a)) is sym.S.true

True

In [10]:
a, b, c, d = sym.symbols('a b c d', real=True)
mat = sym.Matrix([[a,b],[c,d]])
scale_y = mat[:, 0].norm()
mat = mat/scale_y
flip = mat[0,0]*mat[1,1] - mat[0,1]*mat[1,0]
mat[:,0] = mat[:,0]*flip
#sym.solve([expr, sym.sin(sym.atan2(-mat[1,0], mat[0,0]))-sym.sin(sym.atan2(mat[0,1],mat[1,1])), sym.cos(sym.atan2(-mat[1,0], mat[0,0]))-sym.cos(sym.atan2(mat[0,1],mat[1,1]))], [c, d], dict=True)

In [11]:
def derive(
    params: Parameters4DSTEM,
    overfocus: sym.Basic | None = None,  # m
    scan_pixel_pitch: sym.Basic | None = None,  # m
    scan_center: PixelYX | None = None,
    scan_rotation: sym.Basic | None = None,  # rad
    camera_length: sym.Basic | None = None,  # m
    detector_pixel_pitch: sym.Basic | None = None,  # m
    detector_center: PixelYX | None = None,
    detector_rotation: sym.Basic | None = None,  # rad
    semiconv: sym.Basic | None = None,  # rad
    flip_y: sym.logic.boolalg.Boolean | None = None,
    flip_factor: sym.Basic | None = None,
    descan_error: DescanError | None = None,
) -> "Parameters4DSTEM":
    if flip_factor is not None:
        assert flip_y is None
    if flip_y is not None:
        flip_factor = -1. if flip_y else 1.
            
    self = params
    
    return Parameters4DSTEM(
        overfocus=overfocus if overfocus is not None else self.overfocus,
        scan_pixel_pitch=(
            scan_pixel_pitch
            if scan_pixel_pitch is not None
            else self.scan_pixel_pitch
        ),
        scan_center=scan_center if scan_center is not None else self.scan_center,
        scan_rotation=scan_rotation
        if scan_rotation is not None
        else self.scan_rotation,
        camera_length=camera_length
        if camera_length is not None
        else self.camera_length,
        detector_pixel_pitch=(
            detector_pixel_pitch
            if detector_pixel_pitch is not None
            else self.detector_pixel_pitch
        ),
        detector_center=(
            detector_center if detector_center is not None else self.detector_center
        ),
        detector_rotation=(
            detector_rotation
            if detector_rotation is not None
            else self.detector_rotation
        ),
        semiconv=semiconv if semiconv is not None else self.semiconv,
        flip_factor=flip_factor if flip_factor is not None else self.flip_factor,
        descan_error=descan_error
        if descan_error is not None
        else self.descan_error,
    )

In [12]:
def adjust_scan_rotation(params, scan_rotation) -> "Parameters4DSTEM":
    self = params
    de = self.descan_error
    angle = scan_rotation - self.scan_rotation

    # Rotate the input direction
    pxo_pyi, pxo_pxi = rotate(angle).multiply(sym.Matrix([de.pxo_pyi, de.pxo_pxi]))  # add simplificatoin? dotprodsimp=True
    pyo_pyi, pyo_pxi = rotate(angle).multiply(sym.Matrix([de.pyo_pyi, de.pyo_pxi]))
    sxo_pyi, sxo_pxi = rotate(angle).multiply(sym.Matrix([de.sxo_pyi, de.sxo_pxi]))
    syo_pyi, syo_pxi = rotate(angle).multiply(sym.Matrix([de.syo_pyi, de.syo_pxi]))
    new_de = DescanError(
        pxo_pyi=pxo_pyi,
        pyo_pyi=pyo_pyi,
        pxo_pxi=pxo_pxi,
        pyo_pxi=pyo_pxi,
        sxo_pyi=sxo_pyi,
        syo_pyi=syo_pyi,
        sxo_pxi=sxo_pxi,
        syo_pxi=syo_pxi,
        offpxi=de.offpxi,
        offpyi=de.offpyi,
        offsxi=de.offsxi,
        offsyi=de.offsyi,
    )
    return self.derive(
        scan_rotation=scan_rotation,
        descan_error=new_de,
    )

In [13]:
def adjust_scan_pixel_pitch(params, scan_pixel_pitch: sym.Basic) -> "Parameters4DSTEM":
    self = params
    
    de = self.descan_error
    ratio = self.scan_pixel_pitch / scan_pixel_pitch

    new_de = DescanError(
        pxo_pyi=de.pxo_pyi * ratio,
        pyo_pyi=de.pyo_pyi * ratio,
        pxo_pxi=de.pxo_pxi * ratio,
        pyo_pxi=de.pyo_pxi * ratio,
        sxo_pyi=de.sxo_pyi * ratio,
        syo_pyi=de.syo_pyi * ratio,
        sxo_pxi=de.sxo_pxi * ratio,
        syo_pxi=de.syo_pxi * ratio,
        offpxi=de.offpxi,
        offpyi=de.offpyi,
        offsxi=de.offsxi,
        offsyi=de.offsyi,
    )
    return self.derive(
        scan_pixel_pitch=scan_pixel_pitch,
        descan_error=new_de,
    )

In [14]:
def symbol_maker(params_cls, postfix, recurse_for=tuple()):
    symbols_dict = {}
    for attr in params_cls.__annotations__.keys():
        cls = params_cls.__annotations__[attr]
        if cls in recurse_for:
            symbols_dict[attr] = symbol_maker(cls, postfix, recurse_for)
        else:
            symbols_dict[attr] = sym.symbols(f"{attr}_{postfix}")
    return params_cls(**symbols_dict)

In [15]:
p = symbol_maker(Parameters4DSTEM, 'new', recurse_for=[DescanError, PixelYX])

In [16]:
params = Parameters4DSTEM(
        overfocus=0.7,
        scan_pixel_pitch=0.005,
        scan_center=PixelYX(y=17, x=13),
        scan_rotation=1.234,
        camera_length=2.3,
        detector_pixel_pitch=0.0247,
        detector_center=PixelYX(y=11, x=19),
        detector_rotation=2.134,
        semiconv=0.023,
        flip_factor=-1.,
        descan_error=DescanError(offpxi=.345, pxo_pxi=948)
    )
mod= Model4DSTEM.build(params=params, scan_pos=PixelYX(y=13, x=7))

desc = mod._mk_adjust_scan_pixel_pitch()(mod, 2.)
desc

Model4DSTEM(source=PointSource(z=0, semi_conv=0.023, offset_xy=CoordsXY(x=0.0, y=0.0)), scanner=Scanner(z=0.7, scan_pos_x=3.58496437813631, scan_pos_y=-13.9695393770694, scan_tilt_x=0.0, scan_tilt_y=0.0), specimen=Plane(z=0.7), descanner=Descanner(z=0.7, scan_pos_x=3.58496437813631, scan_pos_y=-13.9695393770694, scan_tilt_x=0.0, scan_tilt_y=0.0, descan_error=DescanError(pxo_pxi=2.37000000000000, pxo_pyi=0, pyo_pxi=0, pyo_pyi=0, sxo_pxi=0, sxo_pyi=0, syo_pxi=0, syo_pyi=0, offpxi=0.345, offpyi=0.0, offsxi=0.0, offsyi=0.0)), detector=Plane(z=3.0), _scan_to_real=Matrix([
[ 0.66093021614346, 1.88763641874927],
[-1.88763641874927, 0.66093021614346]]), _real_to_scan=Matrix([
[0.165232554035865, -0.471909104687317],
[0.471909104687317,  0.165232554035865]]), _detector_to_real=Matrix([
[-0.0247,      0],
[      0, 0.0247]]), _real_to_detector=Matrix([
[-40.4858299595142,                0],
[                0, 40.4858299595142]]), scan_center=PixelYX(y=17, x=13), detector_center=PixelYX(y=11, x=

In [17]:
def adjust_detector_pixel_pitch(
        params, detector_pixel_pitch: float
    ) -> "Parameters4DSTEM":
        de = params.descan_error
        ratio = detector_pixel_pitch / params.detector_pixel_pitch

        new_de = DescanError(
            pxo_pyi=de.pxo_pyi * ratio,
            pyo_pyi=de.pyo_pyi * ratio,
            pxo_pxi=de.pxo_pxi * ratio,
            pyo_pxi=de.pyo_pxi * ratio,
            sxo_pyi=de.sxo_pyi * ratio,
            syo_pyi=de.syo_pyi * ratio,
            sxo_pxi=de.sxo_pxi * ratio,
            syo_pxi=de.syo_pxi * ratio,
            offpxi=de.offpxi * ratio,
            offpyi=de.offpyi * ratio,
            offsxi=de.offsxi * ratio,
            offsyi=de.offsyi * ratio,
        )
        return params.derive(
            detector_pixel_pitch=detector_pixel_pitch,
            descan_error=new_de,
        )

In [18]:
adjust_detector_pixel_pitch(params, 2.)

Parameters4DSTEM(overfocus=0.7, scan_pixel_pitch=0.005, scan_center=PixelYX(y=17, x=13), scan_rotation=1.234, camera_length=2.3, detector_pixel_pitch=2.0, detector_center=PixelYX(y=11, x=19), semiconv=0.023, flip_factor=-1.0, descan_error=DescanError(pxo_pxi=76761.13360323886, pxo_pyi=0.0, pyo_pxi=0.0, pyo_pyi=0.0, sxo_pxi=0.0, sxo_pyi=0.0, syo_pxi=0.0, syo_pyi=0.0, offpxi=27.935222672064775, offpyi=0.0, offsxi=0.0, offsyi=0.0), detector_rotation=2.134, xp=<module 'numpy' from '/home/usoltceva/miniforge3/envs/py313/lib/python3.13/site-packages/numpy/__init__.py'>)

In [19]:
mod= Model4DSTEM.build(params=Parameters4DSTEM(
        overfocus=0.7,
        scan_pixel_pitch=0.005,
        scan_center=PixelYX(y=17, x=13),
        scan_rotation=1.234,
        camera_length=2.3,
        detector_pixel_pitch=0.0247,
        detector_center=PixelYX(y=11, x=19),
        detector_rotation=2.134,
        semiconv=0.023,
        flip_factor=-1.,
        descan_error=DescanError(offpxi=.345, pxo_pxi=948)
    ), scan_pos=PixelYX(y=13, x=7))
m = _mk_adjust_detector_pixel_pitch()(mod, 2.)
#m.params
mod

NameError: name '_mk_adjust_detector_pixel_pitch' is not defined

In [20]:
m

m

In [21]:
(sym.Or(sym.And(sympy_equals(mat[0, 0], mat[1, 1]), sympy_equals(mat[0, 1], -mat[1, 0])),
              sym.And(sympy_equals(mat[0, 0], -mat[1, 1]), sympy_equals(mat[0, 1], mat[1, 0])))
            is sym.S.true)

False

In [22]:
sympy_equals(mat[0, 0], mat[1, 1])

False

In [23]:
sympy_equals(mat[0, 0], -mat[1, 1])

False

In [24]:
    def adjust_scan_rotation(params, scan_rotation: float) -> "Parameters4DSTEM":
        self = params
        de = self.descan_error
        angle = scan_rotation - self.scan_rotation

        xp = self.xp

        # Rotate the input direction
        pxo_pyi, pxo_pxi = rotate(angle) @ xp.array((de.pxo_pyi, de.pxo_pxi))
        pyo_pyi, pyo_pxi = rotate(angle) @ xp.array((de.pyo_pyi, de.pyo_pxi))
        sxo_pyi, sxo_pxi = rotate(angle) @ xp.array((de.sxo_pyi, de.sxo_pxi))
        syo_pyi, syo_pxi = rotate(angle) @ xp.array((de.syo_pyi, de.syo_pxi))
        new_de = DescanError(
            pxo_pyi=pxo_pyi,
            pyo_pyi=pyo_pyi,
            pxo_pxi=pxo_pxi,
            pyo_pxi=pyo_pxi,
            sxo_pyi=sxo_pyi,
            syo_pyi=syo_pyi,
            sxo_pxi=sxo_pxi,
            syo_pxi=syo_pxi,
            offpxi=de.offpxi,
            offpyi=de.offpyi,
            offsxi=de.offsxi,
            offsyi=de.offsyi,
        )
        return self.derive(
            scan_rotation=scan_rotation,
            descan_error=new_de,
        )

In [25]:
adjust_scan_rotation(params, 1).descan_error

DescanError(pxo_pxi=922.163869725063, pxo_pyi=-219.813096456279, pyo_pxi=0, pyo_pyi=0, sxo_pxi=0, sxo_pyi=0, syo_pxi=0, syo_pyi=0, offpxi=0.345, offpyi=0.0, offsxi=0.0, offsyi=0.0)

In [26]:
par_sympy = symbol_maker(Parameters4DSTEM, 'sym', [DescanError, PixelYX])
scan_rotation = sym.Symbol('scan_rotation')
adjust_scan_rotation(par_sympy, scan_rotation)

AttributeError: 'Symbol' object has no attribute 'array'

In [27]:
    def adjust_scan_rotation(params, scan_rotation: float) -> "Parameters4DSTEM":
        self = params
        de = self.descan_error
        angle = scan_rotation - self.scan_rotation

        xp = self.xp

        # Rotate the input direction
        pxo_pyi, pxo_pxi = rotate(angle) @ sym.Matrix((de.pxo_pyi, de.pxo_pxi))
        pyo_pyi, pyo_pxi = rotate(angle) @ sym.Matrix((de.pyo_pyi, de.pyo_pxi))
        sxo_pyi, sxo_pxi = rotate(angle) @ sym.Matrix((de.sxo_pyi, de.sxo_pxi))
        syo_pyi, syo_pxi = rotate(angle) @ sym.Matrix((de.syo_pyi, de.syo_pxi))
        new_de = DescanError(
            pxo_pyi=pxo_pyi,
            pyo_pyi=pyo_pyi,
            pxo_pxi=pxo_pxi,
            pyo_pxi=pyo_pxi,
            sxo_pyi=sxo_pyi,
            syo_pyi=syo_pyi,
            sxo_pxi=sxo_pxi,
            syo_pxi=syo_pxi,
            offpxi=de.offpxi,
            offpyi=de.offpyi,
            offsxi=de.offsxi,
            offsyi=de.offsyi,
        )
        return self.derive(
            scan_rotation=scan_rotation,
            descan_error=new_de,
        )

In [28]:
par_sympy = symbol_maker(Parameters4DSTEM, 'old', [DescanError, PixelYX])
scan_rotation_new = sym.Symbol('scan_rotation_new')
adjust_scan_rotation(par_sympy, scan_rotation_new).descan_error._asdict()

{'pxo_pxi': pxo_pxi_old*cos(scan_rotation_new - scan_rotation_old) - pxo_pyi_old*sin(scan_rotation_new - scan_rotation_old),
 'pxo_pyi': pxo_pxi_old*sin(scan_rotation_new - scan_rotation_old) + pxo_pyi_old*cos(scan_rotation_new - scan_rotation_old),
 'pyo_pxi': pyo_pxi_old*cos(scan_rotation_new - scan_rotation_old) - pyo_pyi_old*sin(scan_rotation_new - scan_rotation_old),
 'pyo_pyi': pyo_pxi_old*sin(scan_rotation_new - scan_rotation_old) + pyo_pyi_old*cos(scan_rotation_new - scan_rotation_old),
 'sxo_pxi': sxo_pxi_old*cos(scan_rotation_new - scan_rotation_old) - sxo_pyi_old*sin(scan_rotation_new - scan_rotation_old),
 'sxo_pyi': sxo_pxi_old*sin(scan_rotation_new - scan_rotation_old) + sxo_pyi_old*cos(scan_rotation_new - scan_rotation_old),
 'syo_pxi': syo_pxi_old*cos(scan_rotation_new - scan_rotation_old) - syo_pyi_old*sin(scan_rotation_new - scan_rotation_old),
 'syo_pyi': syo_pxi_old*sin(scan_rotation_new - scan_rotation_old) + syo_pyi_old*cos(scan_rotation_new - scan_rotation_old),


In [29]:
par_sympy

Parameters4DSTEM(overfocus=overfocus_old, scan_pixel_pitch=scan_pixel_pitch_old, scan_center=PixelYX(y=y_old, x=x_old), scan_rotation=scan_rotation_old, camera_length=camera_length_old, detector_pixel_pitch=detector_pixel_pitch_old, detector_center=PixelYX(y=y_old, x=x_old), semiconv=semiconv_old, flip_factor=flip_factor_old, descan_error=DescanError(pxo_pxi=pxo_pxi_old, pxo_pyi=pxo_pyi_old, pyo_pxi=pyo_pxi_old, pyo_pyi=pyo_pyi_old, sxo_pxi=sxo_pxi_old, sxo_pyi=sxo_pyi_old, syo_pxi=syo_pxi_old, syo_pyi=syo_pyi_old, offpxi=offpxi_old, offpyi=offpyi_old, offsxi=offsxi_old, offsyi=offsyi_old), detector_rotation=detector_rotation_old, xp=xp_old)

In [30]:
def _mk_adjust_scan_rotation(cls=Model4DSTEM):
    params_old = symbol_maker(Parameters4DSTEM, 'old', recurse_for=[DescanError, PixelYX])
    params_new_tmp = symbol_maker(Parameters4DSTEM, 'new', recurse_for=[DescanError, PixelYX])

    params_new = params_old.derive(
        descan_error=params_new_tmp.descan_error,
        scan_rotation=params_new_tmp.scan_rotation
    )

    def detector_px(m: Model4DSTEM):
        ray = m.make_source_ray(
            source_dy=0.,
            source_dx=0.,
        ).ray
        data = m.trace(ray)
        return data['detector'].sampling['detector_px']
    
    scan_pos_y, scan_pos_x = sym.symbols('scan_pos_y scan_pos_x')
    scan_pos = PixelYX(scan_pos_y, scan_pos_x)

    model_old = cls.build(params=params_old, scan_pos=scan_pos)
    model_new = cls.build(params=params_new, scan_pos=scan_pos)

    det_px_old = detector_px(model_old)
    det_px_new = detector_px(model_new)

    expanded_expr_y = sym.expand(det_px_old.y - det_px_new.y)
    expanded_expr_x = sym.expand(det_px_old.x - det_px_new.x)

    collected_y = sym.Poly(expanded_expr_y, params_old.camera_length, scan_pos_y, scan_pos_x)
    collected_x = sym.Poly(expanded_expr_x, params_old.camera_length, scan_pos_y, scan_pos_x)
    
    coeff = collected_y.coeffs() + collected_x.coeffs()

    solution = sym.solve([eq for eq in coeff], [*params_new.descan_error],
                        simplify=True)

    def apply_function(m: Model4DSTEM, scan_rotation) -> Model4DSTEM:
        substituted = sym.lambdify(
            [
                *params_old.descan_error,
                params_old.scan_rotation,
                params_new.scan_rotation
            ], [solution[attr] for attr in [
                *params_new.descan_error
            ]])
        
        evaluated = substituted(
            *m.descanner.descan_error,
            float(m.params.scan_rotation), scan_rotation
        )
        new_de = DescanError(*evaluated)
        new_params = m.params.derive(descan_error=new_de, scan_rotation=scan_rotation)
        new_model = cls.build(new_params, m.scan_pos)
        return new_model

    return apply_function

In [31]:
mod= Model4DSTEM.build(params=Parameters4DSTEM(
        overfocus=0.7,
        scan_pixel_pitch=0.005,
        scan_center=PixelYX(y=17, x=13),
        scan_rotation=1.234,
        camera_length=2.3,
        detector_pixel_pitch=0.0247,
        detector_center=PixelYX(y=11, x=19),
        detector_rotation=2.134,
        semiconv=0.023,
        flip_factor=-1.,
        descan_error=DescanError(offpxi=.345, pxo_pxi=948)
    ), scan_pos=PixelYX(y=13, x=7))
_mk_adjust_scan_rotation()(mod, 1.).params

ValueError: y scale 0.0247000000000000 and x scale 0.0247000000000000 are different or rotation angle 1 0 and rotation angle 2 0 are inconsistent.

In [ ]:
adjust_scan_rotation(params, 1)

In [32]:
a = sym.symbols('a_0:10')
a

(a_0, a_1, a_2, a_3, a_4, a_5, a_6, a_7, a_8, a_9)

In [33]:
a, b = sym.symbols('b a')

In [34]:
a

b

In [35]:
isinstance(a, sym.Symbol)

True

In [36]:
from sympy.abc import c

In [37]:
isinstance(c, sym.Symbol)

True

In [38]:
isinstance(sym.Float(1.0), sym.Number)

True

In [39]:
isinstance(sym.Rational(1,2), sym.Number)

True

In [40]:
a = 1

In [44]:
isinstance(1, sym.NumberSymbol)

False

In [46]:
sym.sympify('2^a')

2**a